# SE4050 Deep Learning 2026 - VGG16 Road-Damage Classification

**Task:** multi-label image classification of road-damage types
**(classes: `D00`, `D10`, `D20`, `D40`)** using a pretrained VGG16 backbone.

**Dataset:** road-surface image patches from the RDD-style road damage dataset
(Czech / India / Japan sources, VOC-format annotations, `label_map.pbtxt`
defines D00/D10/D20/D40). Shipped in this repository under `dataset/`.

**Why this task:** the dataset ships bounding-box annotations for damage
categories; an image-level multi-label formulation keeps every annotated image,
matches the four-class label map, and supports evidence-based model comparison
against the other architectures of the group (Custom CNN, ResNet50, ...).

**Design decisions (recorded for the report):**

| Decision | Choice | Reason |
|---|---|---|
| Task | multi-label (sigmoid + BCE) | road patches frequently contain several damage types (1300/4295 images) |
| Labeled data | train/Czech + train/India xmls (4295 images) | train/Japan labels lost in truncated upstream archive; test1/test2 ship without damage annotations |
| Split | 70 / 15 / 15, cluster-grouped | near-duplicate patches can never cross splits (leakage prevention) |
| Backbone | ImageNet-pretrained VGG16, frozen | small dataset, no GPU available (TF has no native-Windows GPU >= 2.11) |
| Training | precomputed bottleneck features + trainable head | reproducible on CPU in minutes |
| Test set | touched **once**, at the end | assignment requirement |

Every hyper-parameter comes from `.env` via `src/config.py`.

## 0. Setup, configuration and reproducibility

In [ ]:
import json
import os
import sys
import time
from pathlib import Path

# Make the project root the working directory regardless of where the kernel
# was started (repo root or notebooks/).
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score, roc_curve)

from src.config import Config
from src.utils import Timer, apply_style, load_json, save_json, set_global_seed

set_global_seed(Config.seed)
apply_style()
Config.ensure_dirs()

RUN = {}          # measured quantities collected during the run

print("TensorFlow :", tf.__version__)
print("NumPy      :", np.__version__)
print("seed       :", Config.seed)
print("classes    :", Config.class_names)
print("image size :", Config.img_size)
print("split      : test", Config.test_size, "val", Config.val_size)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
print("\nConfiguration")
print(json.dumps(Config.as_dict(), indent=2))

## 1. Dataset description, validation and EDA

The validator walks **every** split, opens **every** image (corruption check),
hashes all files (exact duplicates), parses the VOC annotations and produces
the class statistics used in the report.

In [ ]:
from src.validate_dataset import main as validate_dataset

stats = validate_dataset()
RUN["dataset"] = {
    "total_images": stats["total_images"],
    "per_split": stats["per_split"],
    "corrupt": len(stats["corrupt_images"]),
    "exact_duplicate_groups": len(stats["exact_duplicate_groups"]),
}

In [ ]:
print("images per split:", stats["per_split"])
print("image sizes     :", stats["image_sizes"])
print("corrupt images  :", stats["corrupt_images"])
print("duplicate groups:", len(stats["exact_duplicate_groups"]))
print()
print("Damage-label table (train/Czech + train/India):")
for k, v in stats["labels"].items():
    print(f"  {k}: {v}")

In [ ]:
from IPython.display import Image as IPyImage, display

for fig in ("class_distribution_splits.png",
            "damage_class_distribution.png",
            "labels_per_image.png",
            "class_distribution_by_source.png"):
    display(IPyImage(filename=str(Config.results_dir / "figures" / fig)))

## 2. Label construction and leakage-safe splitting

* keep images with **at least one** of the four label-map classes;
* compute a 64-bit perceptual dHash for every image and union near-identical
  images (Hamming <= 4) into clusters;
* assign whole clusters to **train / val / test** with per-class targets
  70 / 15 / 15, so no near-duplicate can leak across splits;
* the saved `split_manifest.csv` is shared by **all** group models.

In [ ]:
from src.dataset import create_or_load_split, leakage_report, load_xy

with Timer():
    labels_df = create_or_load_split(force=True)

leak = leakage_report(labels_df)
print("leakage report:", leak)
RUN["leakage"] = leak
assert leak["passed"], "split leakage detected"

print("\nimages per split:")
print(labels_df["split"].value_counts())
print("\nlabel counts per split (multi-label):")
print(labels_df.groupby("split")[Config.class_names].sum())
print("\nsource per split:")
print(pd.crosstab(labels_df["split"], labels_df["source"]))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
split_cls = labels_df.groupby("split")[Config.class_names].sum().loc[["train", "val", "test"]]
split_cls.plot(kind="bar", ax=ax, rot=0, colormap="tab10")
ax.set_title("Label distribution across train / val / test")
ax.set_ylabel("images containing class")
plt.tight_layout()
plt.savefig(Config.results_dir / "figures" / "split_label_distribution.png")
plt.close()

## 3. Preprocessing

| Step | Applied to | Justification |
|---|---|---|
| Resize to 224 x 224 (bilinear) | all splits | VGG16 native input size; sources ship 600/720/1024 px patches |
| `vgg16.preprocess_input` (RGB -> BGR, channel mean subtraction) | all splits | the exact normalization the ImageNet weights were trained with |
| No random augmentation | train | the backbone is frozen and features are precomputed once, so random transforms would not reach the model; documented as a limitation |
| No test-time statistics | test | test stays unseen; all decisions (thresholds) use **validation only** |
| No class re-weighting | train | imbalance is handled by per-class threshold tuning on validation (section 7) |

**Data quality issues found:** 1 truncated image (`train/Japan`), 0 byte-level
duplicates, 82 near-duplicate clusters (kept intact within one split),
`train.tar.gz` truncated upstream (labels for train/Japan and both test sets
lost) - reported, never silently patched.

## 4. VGG16 model architecture

**Backbone** - `keras.applications.VGG16` with ImageNet weights
(`include_top=False`, `pooling='avg'`):

* 13 convolutional layers in 5 blocks (`2x64 - 2x128 - 3x256 - 3x512 - 3x512`),
  each block followed by 2x2 max pooling; ~14.7 M parameters;
* **entire backbone frozen** (feature extractor only) - protects the small
  dataset from catastrophic over-fitting and makes CPU training feasible;
* global average pooling -> 512-d feature vector per image.

**Classification head** (the trainable part):

| Layer | Units | Activation | Notes |
|---|---|---|---|
| Input | 512 | - | pooled VGG16 features |
| Dense | 256 | ReLU | learned task-specific combination |
| Dropout | - | - | rate 0.5, regularisation |
| Dense (output) | 4 | Sigmoid | independent probability per damage class |

**Training configuration:** binary cross-entropy (multi-label), Adam
(lr = 1e-3, reduced on plateau), batch size 64, up to 100 epochs with early
stopping on validation AUC.

In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers, models

# ---- frozen ImageNet backbone --------------------------------------------
backbone = VGG16(
    include_top=False,
    weights="imagenet",
    pooling="avg",
    input_shape=(Config.img_size, Config.img_size, 3),
)
backbone.trainable = False

# ---- trainable classification head (shared weights in both models) -------
head = models.Sequential(
    [
        layers.Input(shape=(512,), name="vgg16_features"),
        layers.Dense(256, activation="relu", name="fc1"),
        layers.Dropout(Config.dropout, name="dropout"),
        layers.Dense(Config.num_classes, activation="sigmoid", name="damage"),
    ],
    name="classification_head",
)

# ---- full model (backbone + head) for architecture display & inference ---
inp = layers.Input(shape=(Config.img_size, Config.img_size, 3), name="image")
out = head(backbone(inp))
model = models.Model(inp, out, name="VGG16_multilabel")

for m in (model, head):
    m.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=Config.head_lr),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name="binary_acc"),
            tf.keras.metrics.AUC(name="auc", multi_label=True),
        ],
    )

total_params = model.count_params()
trainable = int(np.sum([np.prod(w.shape) for w in model.trainable_weights]))
frozen = total_params - trainable
RUN["model"] = {
    "total_params": total_params,
    "trainable_params": trainable,
    "frozen_params": frozen,
}
print(f"total params     : {total_params:,}")
print(f"trainable params : {trainable:,}")
print(f"frozen params    : {frozen:,}")
model.summary()

HEAD_WEIGHTS = Config.checkpoint_path.with_name("vgg16_head.weights.h5")
FULL_MODEL = Config.checkpoint_path  # results/models/vgg16_multilabel.keras

## 5. Feature extraction (frozen backbone)

One forward pass per image, cached to `results/models/vgg16_features.npz`
keyed by the exact image list - a re-run re-uses the cache only if the data,
image size and backbone are unchanged.

In [ ]:
from tensorflow.keras.applications.vgg16 import preprocess_input

def load_batch(paths, size):
    '''Read images, resize to VGG16 input, apply VGG16 normalization.'''
    batch = np.empty((len(paths), size, size, 3), dtype=np.float32)
    for i, p in enumerate(paths):
        with open(p, "rb") as f:
            data = f.read()
        img = tf.io.decode_image(data, channels=3, expand_animations=False)
        img = tf.image.resize(img, (size, size), method="bilinear")
        batch[i] = img.numpy()
    return preprocess_input(batch)


EXTRACT_LOG = {}   # cache file name -> seconds actually spent extracting


def extract_features(paths, backbone, size, cache_path):
    '''Return (N, 512) features; cache hit requires identical path list.'''
    key = hashlib_md5("\n".join(paths) + f"|{size}|vgg16")
    if cache_path.exists():
        cached = np.load(cache_path, allow_pickle=False)
        if str(cached["key"]) == key and len(cached["features"]) == len(paths):
            elapsed = float(cached["elapsed"]) if "elapsed" in cached.files else 0.0
            EXTRACT_LOG[cache_path.name] = elapsed
            print(f"feature cache hit: {cache_path.name} "
                  f"(original extraction {elapsed:.0f}s)")
            return cached["features"].astype(np.float32)
        print("feature cache stale -> re-extracting")
    feats = []
    t0 = time.time()
    bs = Config.batch_size
    for start in range(0, len(paths), bs):
        chunk = paths[start : start + bs]
        feats.append(backbone.predict(load_batch(chunk, size), verbose=0))
        done = start + len(chunk)
        if done % (bs * 10) == 0 or done == len(paths):
            print(f"  {done}/{len(paths)} images  ({time.time()-t0:.0f}s)", end="\r")
    feats = np.vstack(feats).astype(np.float32)
    elapsed = time.time() - t0
    EXTRACT_LOG[cache_path.name] = elapsed
    np.savez_compressed(cache_path, key=np.str_(key), features=feats,
                        elapsed=np.float64(elapsed))
    print(f"\nextracted {feats.shape} in {elapsed:.0f}s -> {cache_path.name}")
    return feats


def hashlib_md5(text):
    import hashlib
    return hashlib.md5(text.encode("utf-8")).hexdigest()

In [ ]:
train_paths, y_train = load_xy(labels_df, "train")
val_paths, y_val = load_xy(labels_df, "val")
test_paths, y_test = load_xy(labels_df, "test")
print(len(train_paths), len(val_paths), len(test_paths))

X_train = extract_features(train_paths, backbone, Config.img_size, Config.features_cache)
X_val = extract_features(val_paths, backbone, Config.img_size, Config.features_cache.with_name(Config.features_cache.stem + "_val.npz"))
X_test = extract_features(test_paths, backbone, Config.img_size, Config.features_cache.with_name(Config.features_cache.stem + "_test.npz"))

RUN["feature_seconds"] = round(sum(EXTRACT_LOG.values()), 1)
print("feature extraction (sum over splits):", RUN["feature_seconds"], "s")
for name, secs in EXTRACT_LOG.items():
    print(f"  {name}: {secs:.1f}s")
print("features:", X_train.shape, X_val.shape, X_test.shape)

## 6. Training the classification head

* loss `binary_crossentropy`, optimizer `Adam(lr=1e-3)`,
* metrics: binary accuracy + **multi-label ROC-AUC**,
* callbacks: `EarlyStopping(patience=12, restore_best)`,
  `ReduceLROnPlateau(patience=5, factor=0.5)`, custom best-val-AUC weights
  checkpoint (`BestHeadWeights`),
* single seed (`Config.seed`) for Python / NumPy / TensorFlow.

In [ ]:
run_meta = {
    "config": Config.as_dict(),
    "split_md5": hashlib_md5("\n".join(train_paths + val_paths + test_paths)),
    "feature_key": hashlib_md5(str(Config.img_size) + "vgg16"),
}
meta_path = Config.checkpoint_path.with_name("run_meta.json")

train_skipped = False
if (FULL_MODEL.exists() and HEAD_WEIGHTS.exists() and meta_path.exists()
        and Config.history_path.exists()):
    previous = load_json(meta_path)
    if previous == run_meta:
        train_skipped = True
        print("Valid checkpoint + matching config found -> skipping training.")

history = None
if not train_skipped:
    class BestHeadWeights(tf.keras.callbacks.Callback):
        '''Saves head weights whenever the monitored metric improves.'''
        def __init__(self, path, monitor="val_auc"):
            super().__init__()
            self.path = str(path)
            self.monitor = monitor
            self.best = -np.inf
        def on_epoch_end(self, epoch, logs=None):
            value = (logs or {}).get(self.monitor)
            if value is not None and value > self.best:
                self.best = float(value)
                self.model.save_weights(self.path)

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_auc", mode="max",
            patience=Config.patience_earlystop, restore_best_weights=True,
            verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_auc", mode="max",
            factor=0.5, patience=Config.patience_lr, min_lr=1e-6, verbose=1),
        BestHeadWeights(HEAD_WEIGHTS, monitor="val_auc"),
    ]

    with Timer() as train_timer:
        history = head.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=Config.head_epochs,
            batch_size=Config.batch_size,
            shuffle=True,
            callbacks=callbacks,
            verbose=2,
        )
    RUN["train_seconds"] = round(train_timer.elapsed, 1)
    hist = {k: [float(v) for v in vals] for k, vals in history.history.items()}
    save_json(hist, Config.history_path)
    save_json(run_meta, meta_path)
    model.save(str(FULL_MODEL))          # backbone + trained head
    print(f"trained in {RUN['train_seconds']}s, "
          f"best val AUC {max(hist['val_auc']):.4f}")
else:
    hist = load_json(Config.history_path)
    RUN["train_seconds"] = None
    head.load_weights(str(HEAD_WEIGHTS))
    print("loaded history:", {k: round(v[-1], 4) for k, v in hist.items()})

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(hist["loss"], label="train")
axes[0].plot(hist["val_loss"], label="val")
axes[0].set_title("Binary cross-entropy")
axes[1].plot(hist["binary_acc"], label="train")
axes[1].plot(hist["val_binary_acc"], label="val")
axes[1].set_title("Binary accuracy")
axes[2].plot(hist["auc"], label="train")
axes[2].plot(hist["val_auc"], label="val")
axes[2].set_title("Multi-label ROC-AUC")
for ax in axes:
    ax.set_xlabel("epoch")
    ax.legend()
    ax.grid(alpha=0.3)
plt.suptitle("VGG16 head - learning curves")
plt.tight_layout()
plt.savefig(Config.results_dir / "figures" / "learning_curves.png")
plt.show()

## 7. Evaluation on the unseen test set (threshold 0.5)

Metrics: subset accuracy, per-class precision / recall / F1, micro & macro
averages, ROC-AUC (micro, macro and per class). The test set is used **only
here**.

In [ ]:
head.load_weights(str(HEAD_WEIGHTS))
print("loaded best head weights:", HEAD_WEIGHTS.name)

prob_val = head.predict(X_val, verbose=0)
prob_test = head.predict(X_test, verbose=0)
pred_val = (prob_val >= 0.5).astype(int)
pred_test = (prob_test >= 0.5).astype(int)

def multilabel_metrics(y_true, pred, prob, average):
    return {
        "precision": precision_score(y_true, pred, average=average, zero_division=0),
        "recall": recall_score(y_true, pred, average=average, zero_division=0),
        "f1": f1_score(y_true, pred, average=average, zero_division=0),
        "roc_auc": roc_auc_score(y_true, prob, average=average),
    }

metrics = {
    "threshold": 0.5,
    "subset_accuracy": float(accuracy_score(y_test, pred_test)),
    "hamming_accuracy": float((y_test == pred_test).mean()),
    "micro": multilabel_metrics(y_test, pred_test, prob_test, "micro"),
    "macro": multilabel_metrics(y_test, pred_test, prob_test, "macro"),
    "per_class": {},
}
for i, c in enumerate(Config.class_names):
    metrics["per_class"][c] = {
        "precision": float(precision_score(y_test[:, i], pred_test[:, i], zero_division=0)),
        "recall": float(recall_score(y_test[:, i], pred_test[:, i], zero_division=0)),
        "f1": float(f1_score(y_test[:, i], pred_test[:, i], zero_division=0)),
        "roc_auc": float(roc_auc_score(y_test[:, i], prob_test[:, i])),
        "support": int(y_test[:, i].sum()),
    }
save_json(metrics, Config.results_dir / "metrics" / "test_metrics.json")

pd.DataFrame(metrics["per_class"]).T

In [ ]:
print(f"subset accuracy : {metrics['subset_accuracy']:.4f}")
print(f"micro  P/R/F1   : {metrics['micro']['precision']:.3f} / "
      f"{metrics['micro']['recall']:.3f} / {metrics['micro']['f1']:.3f}")
print(f"macro  P/R/F1   : {metrics['macro']['precision']:.3f} / "
      f"{metrics['macro']['recall']:.3f} / {metrics['macro']['f1']:.3f}")
print(f"micro/macro AUC : {metrics['micro']['roc_auc']:.3f} / "
      f"{metrics['macro']['roc_auc']:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
# ROC curves
for i, c in enumerate(Config.class_names):
    fpr, tpr, _ = roc_curve(y_test[:, i], prob_test[:, i])
    axes[0].plot(fpr, tpr, label=f"{c} (AUC {metrics['per_class'][c]['roc_auc']:.3f})")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.5)
axes[0].set_title("ROC curves - test set")
axes[0].set_xlabel("false positive rate")
axes[0].set_ylabel("true positive rate")
axes[0].legend(fontsize=8)
# micro-average ROC
fpr, tpr, _ = roc_curve(y_test.ravel(), prob_test.ravel())
axes[1].plot(fpr, tpr, color="crimson", label=f"micro (AUC {metrics['micro']['roc_auc']:.3f})")
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.5)
axes[1].set_title("Micro-averaged ROC - test set")
axes[1].set_xlabel("false positive rate")
axes[1].set_ylabel("true positive rate")
axes[1].legend()
plt.tight_layout()
plt.savefig(Config.results_dir / "figures" / "roc_curves.png")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9, 8))
for i, (c, ax) in enumerate(zip(Config.class_names, axes.ravel())):
    cm = confusion_matrix(y_test[:, i], pred_test[:, i])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["no " + c, c], yticklabels=["no " + c, c])
    ax.set_title(f"{c}  (F1 {metrics['per_class'][c]['f1']:.3f})")
    ax.set_ylabel("true")
    ax.set_xlabel("predicted")
plt.suptitle("Per-class confusion matrices - test set (threshold 0.5)")
plt.tight_layout()
plt.savefig(Config.results_dir / "figures" / "confusion_matrices.png")
plt.show()

## 8. Decision-threshold tuning (validation only)

A fixed 0.5 threshold is rarely optimal for imbalanced multi-label heads.
For every class we sweep thresholds on the **validation** set and keep the one
maximising F1; the test set is then re-scored with those thresholds.

In [ ]:
grid = np.arange(0.05, 0.95, 0.05)
best_thr = {}
for i, c in enumerate(Config.class_names):
    f1s = [f1_score(y_val[:, i], (prob_val[:, i] >= t).astype(int), zero_division=0)
           for t in grid]
    best_thr[c] = float(grid[int(np.argmax(f1s))])
print("tuned thresholds (from validation):", best_thr)

pred_test_tuned = np.stack(
    [(prob_test[:, i] >= best_thr[c]).astype(int)
     for i, c in enumerate(Config.class_names)], axis=1)

metrics_tuned = {
    "thresholds": best_thr,
    "subset_accuracy": float(accuracy_score(y_test, pred_test_tuned)),
    "micro": multilabel_metrics(y_test, pred_test_tuned, prob_test, "micro"),
    "macro": multilabel_metrics(y_test, pred_test_tuned, prob_test, "macro"),
    "per_class": {},
}
for i, c in enumerate(Config.class_names):
    metrics_tuned["per_class"][c] = {
        "precision": float(precision_score(y_test[:, i], pred_test_tuned[:, i], zero_division=0)),
        "recall": float(recall_score(y_test[:, i], pred_test_tuned[:, i], zero_division=0)),
        "f1": float(f1_score(y_test[:, i], pred_test_tuned[:, i], zero_division=0)),
        "roc_auc": metrics["per_class"][c]["roc_auc"],   # threshold-independent
        "support": int(y_test[:, i].sum()),
    }
save_json(metrics_tuned, Config.results_dir / "metrics" / "test_metrics_tuned.json")
save_json(best_thr, Config.results_dir / "metrics" / "thresholds.json")

cmp_df = pd.DataFrame({
    "F1 @0.5": {c: metrics["per_class"][c]["f1"] for c in Config.class_names},
    "F1 tuned": {c: metrics_tuned["per_class"][c]["f1"] for c in Config.class_names},
    "threshold": best_thr,
})
cmp_df.loc["micro"] = [metrics["micro"]["f1"], metrics_tuned["micro"]["f1"], np.nan]
cmp_df.loc["macro"] = [metrics["macro"]["f1"], metrics_tuned["macro"]["f1"], np.nan]
cmp_df

In [ ]:
print(f"subset accuracy  0.5 -> tuned: "
      f"{metrics['subset_accuracy']:.4f} -> {metrics_tuned['subset_accuracy']:.4f}")
print(f"micro F1         0.5 -> tuned: "
      f"{metrics['micro']['f1']:.4f} -> {metrics_tuned['micro']['f1']:.4f}")
print(f"macro F1         0.5 -> tuned: "
      f"{metrics['macro']['f1']:.4f} -> {metrics_tuned['macro']['f1']:.4f}")

## 9. Error analysis

* per-source (Czech vs India) performance - detects country bias,
* which labels are systematically missed or hallucinated,
* the most confident mistakes, with images.

In [ ]:
# --- per-source breakdown ---------------------------------------------------
rows = []
test_df = labels_df[labels_df["split"] == "test"].reset_index(drop=True)
for source, grp in test_df.groupby("source"):
    idx = grp.index.to_numpy()
    for name, pred in (("0.5", pred_test), ("tuned", pred_test_tuned)):
        rows.append({
            "source": source,
            "threshold": name,
            "n": len(idx),
            "micro_f1": f1_score(y_test[idx], pred[idx], average="micro", zero_division=0),
            "subset_acc": accuracy_score(y_test[idx], pred[idx]),
        })
source_df = pd.DataFrame(rows)
save_json(source_df.to_dict("records"),
          Config.results_dir / "metrics" / "source_metrics.json")
source_df

In [ ]:
# --- label-level error counts (tuned thresholds) ---------------------------
fn = ((y_test == 1) & (pred_test_tuned == 0)).sum(axis=0)
fp = ((y_test == 0) & (pred_test_tuned == 1)).sum(axis=0)
err_df = pd.DataFrame({
    "class": Config.class_names,
    "missed (FN)": fn,
    "spurious (FP)": fp,
    "support": y_test.sum(axis=0).astype(int),
}).set_index("class")
save_json({"fn": dict(zip(Config.class_names, fn.astype(int))),
           "fp": dict(zip(Config.class_names, fp.astype(int)))},
          Config.results_dir / "error_analysis" / "label_errors.json")
err_df

In [ ]:
# --- most confident errors -------------------------------------------------
wrong = (pred_test_tuned != y_test).any(axis=1)
conf_true = np.where(y_test == 1, prob_test, 1 - prob_test).min(axis=1)
error_order = np.argsort(conf_true * wrong)[::-1]
error_order = [i for i in error_order if wrong[i]][:8]

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, i in zip(axes.ravel(), error_order):
    with open(test_paths[i], "rb") as f:
        img = plt.imread(f, format="jpg")
    ax.imshow(img)
    true_lbl = ",".join(c for c, v in zip(Config.class_names, y_test[i]) if v) or "-"
    pred_lbl = ",".join(c for c, v in zip(Config.class_names, pred_test_tuned[i]) if v) or "-"
    ax.set_title(f"true: {true_lbl}\npred: {pred_lbl}  (p={conf_true[i]:.2f})",
                 fontsize=9, color="crimson")
    ax.axis("off")
plt.suptitle("Most confident errors - test set (tuned thresholds)")
plt.tight_layout()
plt.savefig(Config.results_dir / "error_analysis" / "confident_errors.png")
plt.show()

# save full prediction table
pred_table = pd.DataFrame({
    "path": [Path(p).name for p in test_paths],
    "source": test_df["source"],
})
for i, c in enumerate(Config.class_names):
    pred_table[f"true_{c}"] = y_test[:, i]
    pred_table[f"prob_{c}"] = np.round(prob_test[:, i], 4)
    pred_table[f"pred_{c}"] = pred_test_tuned[:, i]
pred_table.to_csv(Config.results_dir / "predictions" / "test_predictions.csv", index=False)
print("saved", len(pred_table), "test predictions")

## 10. Computational considerations

In [ ]:
import os as _os

RUN["computational"] = {
    **RUN.get("model", {}),
    "feature_extraction_seconds": RUN.get("feature_seconds"),
    "training_seconds": RUN.get("train_seconds"),
    "epochs_run": len(hist["loss"]),
    "best_epoch": int(np.argmax(hist["val_auc"])) + 1,
    "checkpoint_mb": round(_os.path.getsize(Config.checkpoint_path) / 1e6, 2),
    "backbone": "VGG16 ImageNet (frozen)",
    "device": "CPU (TensorFlow 2.21, native Windows - no GPU support)",
}
save_json(RUN, Config.results_dir / "metrics" / "run_summary.json")
print(json.dumps(RUN["computational"], indent=2))

## 11. Critical analysis and discussion

> The quantitative summary below is printed from the metrics this run just
> saved to `results/metrics/`, so it always matches the artifacts.

In [ ]:
from IPython.display import display

test_m = load_json(Config.results_dir / "metrics" / "test_metrics.json")
tuned_m = load_json(Config.results_dir / "metrics" / "test_metrics_tuned.json")
thr = load_json(Config.results_dir / "metrics" / "thresholds.json")
src_m = load_json(Config.results_dir / "metrics" / "source_metrics.json")
err_m = load_json(Config.results_dir / "error_analysis" / "label_errors.json")
run_s = load_json(Config.results_dir / "metrics" / "run_summary.json")

print("TEST @ threshold 0.5")
display(pd.DataFrame(test_m["per_class"]).T[["precision", "recall", "f1", "roc_auc", "support"]])
for k in ("micro", "macro"):
    print(f"  {k}: P={test_m[k]['precision']:.3f} R={test_m[k]['recall']:.3f} "
          f"F1={test_m[k]['f1']:.3f} AUC={test_m[k]['roc_auc']:.3f}")
print(f"  subset accuracy: {test_m['subset_accuracy']:.4f}")

print("\nTEST @ validation-tuned thresholds:", thr)
display(pd.DataFrame(tuned_m["per_class"]).T[["precision", "recall", "f1", "roc_auc", "support"]])
for k in ("micro", "macro"):
    print(f"  {k}: F1 {test_m[k]['f1']:.4f} -> {tuned_m[k]['f1']:.4f}")
print(f"  subset accuracy {test_m['subset_accuracy']:.4f} -> {tuned_m['subset_accuracy']:.4f}")

print("\nper-source breakdown (micro F1):")
display(pd.DataFrame(src_m))
print("\nlabel-level error counts @ tuned thresholds (FN / FP / support):")
err_df = pd.DataFrame({
    "missed (FN)": pd.Series(err_m["fn"]),
    "spurious (FP)": pd.Series(err_m["fp"]),
    "support": pd.Series({c: test_m["per_class"][c]["support"]
                          for c in Config.class_names}),
})
display(err_df)
print("\ncompute:", json.dumps(run_s["computational"], indent=2))

### Qualitative discussion

*(filled from the evidence above after execution - see the report for the
full write-up: what works, what does not, error causes, literature links.)*


## 12. Reproducibility

* seed: `Config.seed = 42` (Python / NumPy / TensorFlow),
* split: `results/metrics/split_manifest.csv` (shared by all group models),
* config: `.env` + `src/config.py`, recorded in `results/metrics/run_summary.json`,
* dependencies: `requirements.txt`,
* known limits: TensorFlow oneDNN CPU kernels can introduce tiny numerical
  variation between runs; exact bit-level reproducibility is not claimed.

## 13. Status checklist

| Item | Status |
|---|---|
| VGG16 implementation | done (section 4) |
| Training | done (section 6) |
| Evaluation | done (sections 7-8) |
| Error analysis | done (section 9) |
| Notebook runs end-to-end | verified by execution below |
| Test set untouched until final evaluation | yes |